# Scalars, Vectors & Matrices
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Distinguish** a scalar, a vector, and a matrix and explain what each one represents
- **Describe** a vector as an arrow in space with both direction and magnitude
- **Interpret** a dataset as a matrix where each row is one data point and each column is one feature

> **Tip:** Start by moving the **x and y sliders** in the vector widget and watch the arrow update. The arrow's length is the magnitude and the angle is the direction. Both change as you move the components.

---
## How we got here

This is the first notebook in the linear algebra series. In the calculus series we worked with functions of one or many variables and used the gradient. The gradient is a vector. In the Python notebooks we used NumPy arrays and pandas DataFrames. Both are linear algebra objects. This notebook gives those familiar tools their mathematical names and shows the structure that makes them powerful.

---
## Why this matters for data science

Every dataset is a matrix. Every data point is a vector. Every model's learned parameters are a vector (or a matrix of vectors for neural networks). When scikit-learn's `LinearRegression` computes predictions, it multiplies a matrix of features by a vector of weights. When a neural network does a forward pass, it applies a sequence of matrix multiplications. Linear algebra is not abstract mathematics for data scientists, it is the literal format of everything you work with.

---
## Try it yourself

In [2]:
out = Output()

view_tg = widgets.ToggleButtons(
    options=["Scalar", "Vector", "Matrix"],
    value="Vector",
    style={"button_width": "110px"})

sc_sl = FloatSlider(value=0.87, min=-5.0, max=5.0, step=0.01,
    description="a =", style={"description_width": "40px"},
    layout=widgets.Layout(width="400px"))

vx_sl = FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.1,
    description="x:", style={"description_width": "30px"},
    layout=widgets.Layout(width="380px"))
vy_sl = FloatSlider(value=4.0, min=-5.0, max=5.0, step=0.1,
    description="y:", style={"description_width": "30px"},
    layout=widgets.Layout(width="380px"))

_MATS = {
    "Feature matrix X — 5 samples, 4 features": (
        np.array([[28, 72000, 12, 1],
                  [35, 88000, 7, 1],
                  [22, 45000, 2, 0],
                  [41, 110000, 20, 1],
                  [30, 63000, 9, 0]], dtype=float),
        [f"sample {i}" for i in range(5)],
        ["age", "income", "purchases", "premium"]),
    "Weight matrix W — 4 inputs to 3 neurons": (
        np.array([[0.5, -0.3, 0.8],
                  [-0.2, 0.7, -0.4],
                  [0.1, 0.6, 0.2],
                  [-0.5, 0.3, 0.9]], dtype=float),
        [f"in_{i}" for i in range(4)],
        [f"neuron_{j}" for j in range(3)]),
    "Identity matrix I (3×3)": (
        np.eye(3),
        [f"row {i}" for i in range(3)],
        [f"col {j}" for j in range(3)]),
}
mat_dd = Dropdown(options=list(_MATS.keys()), value=list(_MATS.keys())[0],
    description="Example:", style={"description_width": "80px"},
    layout=widgets.Layout(width="500px"))

ctrl_box = VBox([])

def _col_norm(M):
    """Normalize each column to 0..1 for COLORING only, so columns on very
    different scales (e.g. income vs age) each show their own gradient.
    The real values are still displayed as text."""
    M = np.asarray(M, dtype=float)
    cmin = M.min(axis=0)
    cmax = M.max(axis=0)
    rng = np.where(cmax - cmin == 0, 1.0, cmax - cmin)
    return (M - cmin) / rng

def _caption(html_inner):
    return HTML(
        f'<div style="font-family:{FONT["family"]}; font-size:14px; '
        f'padding:8px 18px; background:#f8f9fa; border-radius:6px; margin-top:4px;">'
        f'{html_inner}</div>')

def render(change=None):
    view = view_tg.value
    with out:
        clear_output(wait=True)
        if view == "Scalar":
            val = sc_sl.value
            fig = go.Figure()
            fig.add_shape(type="line", x0=-5, y0=0, x1=5, y1=0,
                line=dict(color=PALETTE["muted"], width=2))
            for t in range(-5, 6):
                fig.add_shape(type="line", x0=t, y0=-0.06, x1=t, y1=0.06,
                    line=dict(color=PALETTE["muted"], width=1))
                fig.add_annotation(x=t, y=-0.18, text=str(t), showarrow=False,
                    font=dict(size=11, family=FONT["family"]))
            fig.add_trace(go.Scatter(x=[val], y=[0], mode="markers+text",
                marker=dict(color=PALETTE["secondary"], size=18,
                            line=dict(width=2, color="white")),
                text=[f"  a = {val:.2f}"], textposition="top right",
                textfont=dict(size=13, color=PALETTE["secondary"]),
                showlegend=False))
            layout = base_layout(
                title=f"Scalar — a = {val:.2f} — one number, no direction",
                xaxis_title="", yaxis_title="")
            layout.update(
                xaxis=dict(range=[-5.6, 5.6], showgrid=False, showticklabels=False),
                yaxis=dict(range=[-0.5, 0.5], showticklabels=False, showgrid=False),
                height=220)
            fig.update_layout(layout)
            display(go.FigureWidget(fig))
            sign = "positive" if val > 0 else "negative" if val < 0 else "zero"
            display(_caption(
                f'<b>a = {val:.2f}</b>&emsp;|a| = <b>{abs(val):.2f}</b>&emsp;'
                f'sign: <b>{sign}</b>&emsp;'
                f'A scalar has magnitude but no direction — just a position on the line.'))

        elif view == "Vector":
            x, y = vx_sl.value, vy_sl.value
            mag = float(np.sqrt(x**2 + y**2))
            angle_deg = float(np.degrees(np.arctan2(y, x)))
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=[0, x], y=[0, 0], mode="lines",
                line=dict(color=PALETTE["muted"], width=1.5, dash="dot"),
                showlegend=False))
            fig.add_trace(go.Scatter(x=[x, x], y=[0, y], mode="lines",
                line=dict(color=PALETTE["muted"], width=1.5, dash="dot"),
                showlegend=False))
            fig.add_trace(go.Scatter(x=[0, x], y=[0, y], mode="lines",
                line=dict(color=PALETTE["primary"], width=3), showlegend=False))
            fig.add_annotation(x=x, y=y, ax=0, ay=0,
                xref="x", yref="y", axref="x", ayref="y",
                arrowhead=3, arrowsize=1.5, arrowwidth=2.5,
                arrowcolor=PALETTE["primary"], showarrow=True, text="")
            fig.add_trace(go.Scatter(x=[x], y=[y], mode="markers+text",
                marker=dict(color=PALETTE["primary"], size=9,
                            line=dict(width=2, color="white")),
                text=[f"  ({x:.1f}, {y:.1f})"], textposition="top right",
                textfont=dict(size=12, family=FONT["family"]),
                showlegend=False))
            layout = base_layout(
                title=f"Vector v = ({x:.1f}, {y:.1f})  |  ‖v‖ = {mag:.3f}  |  θ = {angle_deg:.1f}°",
                xaxis_title="", yaxis_title="")
            layout.update(
                xaxis=dict(range=[-5.5, 5.5], zeroline=True, zerolinecolor="#ccc"),
                yaxis=dict(range=[-5.5, 5.5], zeroline=True, zerolinecolor="#ccc",
                           scaleanchor="x"),
                height=400)
            fig.update_layout(layout)
            display(go.FigureWidget(fig))
            display(_caption(
                f'<b>v = ({x:.2f}, {y:.2f})</b>&emsp;'
                f'‖v‖ = √({x:.2f}² + {y:.2f}²) = <b>{mag:.4f}</b>&emsp;'
                f'θ = <b>{angle_deg:.1f}°</b>'))

        else:  # Matrix
            M, row_lbl, col_lbl = _MATS[mat_dd.value]
            nr, nc = M.shape
            fig = go.Figure(go.Heatmap(
                z=_col_norm(M), x=col_lbl, y=row_lbl,
                colorscale="Blues", showscale=False, zmin=0, zmax=1,
                text=M.round(2), texttemplate="%{text}", hoverinfo="text"))
            layout = base_layout(
                title=f"Matrix — shape ({nr}, {nc}) — {nr} rows × {nc} columns",
                xaxis_title="", yaxis_title="")
            layout.update(height=min(200 + 55*nr, 450))
            fig.update_layout(layout)
            fig.update_yaxes(autorange="reversed")
            display(go.FigureWidget(fig))
            display(_caption(
                f'Shape <b>({nr}, {nc})</b> — <b>{nr}</b> rows × <b>{nc}</b> columns = '
                f'<b>{nr*nc}</b> numbers. Color is normalized per column so each '
                f'feature shows its own range; the printed values are the real numbers.'))

def on_view_change(change):
    view = view_tg.value
    if view == "Scalar":
        ctrl_box.children = [view_tg, sc_sl, out]
    elif view == "Vector":
        ctrl_box.children = [view_tg, vx_sl, vy_sl, out]
    else:
        ctrl_box.children = [view_tg, mat_dd, out]
    render()

view_tg.observe(on_view_change, names="value")
sc_sl.observe(render, names="value")
vx_sl.observe(render, names="value")
vy_sl.observe(render, names="value")
mat_dd.observe(render, names="value")

ctrl_box.children = [view_tg, vx_sl, vy_sl, out]
display(ctrl_box)
render()

---
## What's happening?

Linear algebra gives us three mathematical objects that data science uses constantly, each one a higher-dimensional version of the previous.

A **scalar** is a single number. Temperature, price, probability, these are scalars. In Python: `x = 3.14`.

A **vector** is an ordered list of numbers. Think of it as a point in space, or an arrow from the origin to that point. Each number in the list is one coordinate. A single customer's features, age, income, purchase count, form a vector.

$$\mathbf{v} = \begin{pmatrix} 3.2 \\ 47000 \\ 12 \end{pmatrix}$$

A **matrix** is a rectangular grid of numbers arranged in rows and columns. A dataset with n samples and p features is an n × p matrix. We write the shape as (rows × columns).

| Object | Shape | Example | ML equivalent |
|--------|-------|---------|--------------|
| Scalar | (single number) | 0.87 (a probability) | A model's accuracy or loss value |
| Vector | (p,) | [age, income, score] | One data point / one row of the dataset |
| Matrix | (n, p) | All n rows of the dataset | The feature matrix X passed to model.fit() |

### NumPy representation

```python
scalar = 3.14                                   # a Python float
vector = np.array([28, 72000, 12])              # shape (3,)
matrix = np.array([[28, 72000, 12],             # shape (4, 3)
                   [35, 88000, 7],
                   [22, 45000, 2],
                   [41, 110000, 20]])
```

Return to the vector widget and set x=3, y=4. Verify that the magnitude is 5 (this is the famous 3-4-5 right triangle, which is also the Pythagorean theorem applied to vector length).

---
## Real-world example: A pandas DataFrame and its NumPy matrix equivalent

The chart below shows the same information in two representations: a pandas DataFrame (the labeled, human-readable view) and a NumPy matrix (the computation-ready view). Every time you call `df.values` or `np.array(df)`, you perform this conversion.

Notice:

- **Notice:** The DataFrame uses column names as labels, which makes it readable; the NumPy matrix uses integer indices [0,0], [0,1], etc., which makes it fast to compute with
- **Notice:** Each row of the matrix is a vector, one customer's features; the entire matrix is the input X that `model.fit(X, y)` receives
- **Notice:** The matrix shape (5, 4) means 5 samples and 4 features; this is what `X.shape` returns in scikit-learn, and what determines the first dimension of the weight vector

> **Discussion question:** A dataset has 10,000 rows and 200 features. What is the shape of the feature matrix X? If you add a bias column of all ones, what does X become? How many numbers are stored in total, and how many bytes does the matrix occupy in memory if stored as float64?

In [4]:
# ── DataFrame and its NumPy equivalent shown side by side ────────────────────
data = {
    "age":          [28, 35, 22, 41, 30],
    "annual_income":[72000, 88000, 45000, 110000, 63000],
    "num_purchases":[12, 7, 2, 20, 9],
    "is_premium":   [1, 1, 0, 1, 0],
}
df = pd.DataFrame(data)
X  = df.values   # shape (5, 4)

def _col_norm(M):
    M = np.asarray(M, dtype=float)
    cmin = M.min(axis=0); cmax = M.max(axis=0)
    rng = np.where(cmax - cmin == 0, 1.0, cmax - cmin)
    return (M - cmin) / rng

fig = make_subplots(rows=1, cols=2,
    subplot_titles=["pandas DataFrame (labeled view)",
                    "NumPy Matrix X, shape (5, 4) (indexed view)"])

# Left: same numbers, labeled by column name and Row i
fig.add_trace(go.Heatmap(
    z=_col_norm(X), x=list(data.keys()),
    y=[f"Row {i}" for i in range(5)],
    colorscale="Blues", showscale=False, zmin=0, zmax=1,
    text=X, texttemplate="%{text}", hoverinfo="text",
), row=1, col=1)

# Right: identical numbers, now addressed purely by integer position
fig.add_trace(go.Heatmap(
    z=_col_norm(X), x=[f"col {j}" for j in range(4)],
    y=[f"row {i}" for i in range(5)],
    colorscale="Oranges", showscale=False, zmin=0, zmax=1,
    text=X, texttemplate="%{text}", hoverinfo="text",
), row=1, col=2)

fig.update_layout(
    title=dict(text="Same Data: pandas DataFrame vs NumPy Matrix",
               font=dict(size=FONT["size_title"])),
    paper_bgcolor=PALETTE["background"], height=300)
fig.update_yaxes(autorange="reversed")
display(go.FigureWidget(fig))

FigureWidget({
    'data': [{'colorscale': [[0.0, 'rgb(247,251,255)'], [0.125,
                             'rgb(222,235,247)'], [0.25, 'rgb(198,219,239)'],
                             [0.375, 'rgb(158,202,225)'], [0.5,
                             'rgb(107,174,214)'], [0.625, 'rgb(66,146,198)'],
                             [0.75, 'rgb(33,113,181)'], [0.875, 'rgb(8,81,156)'],
                             [1.0, 'rgb(8,48,107)']],
              'hoverinfo': 'text',
              'showscale': False,
              'text': {'bdata': ('HAAAAEAZAQAMAAAAAQAAACMAAADAVw' ... 'AAAQAAAB4AAAAY9gAACQAAAAAAAAA='),
                       'dtype': 'i4',
                       'shape': '5, 4'},
              'texttemplate': '%{text}',
              'type': 'heatmap',
              'uid': 'd79cbb37-5ef0-4000-a2d2-60b402247180',
              'x': [age, annual_income, num_purchases, is_premium],
              'xaxis': 'x',
              'y': [Row 0, Row 1, Row 2, Row 3, Row 4],
              'yaxis': 'y

### Object types and their numpy representations

| Object | Shape | numpy representation | ML equivalent |
|--------|-------|---------------------|--------------|
| Scalar | () | `np.float64(3.14)` or just `3.14` | Accuracy, loss, a single prediction |
| Row vector | (1, p) | `np.array([[1, 2, 3]])` | One data point (as a 2D row) |
| Column vector | (p, 1) | `np.array([[1],[2],[3]])` | Weight vector in matrix equations |
| 1D array | (p,) | `np.array([1, 2, 3])` | Weight vector in `dot()` operations |
| Feature matrix | (n, p) | `np.random.randn(n, p)` | Dataset X: n samples, p features |
| Weight matrix | (p, k) | `np.random.randn(p, k)` | Neural network layer weights |

---
## Key takeaway

> **A scalar is one number, a vector is an ordered list of numbers with direction and magnitude, and a matrix is a rectangular table of numbers — every dataset is a matrix, every data point is a vector, and every learned parameter is one or the other.**

---
*Next up: Matrix Operations — the arithmetic of matrices: addition, subtraction, and scalar multiplication, and how they power batch computation in model training*